# LLM: pretraining and SFT

Next-token prediction and response-only supervision.

> All examples run on CPU without model downloads and use synthetic data. They explain mechanisms; real training quality must be evaluated separately.

## What to understand

A causal LM predicts each next token from preceding tokens. Pretraining usually supervises every non-padding token; response-only SFT supervises the assistant response. Shift logits and labels exactly once.

This is a tiny causal-attention model, not a pretrained language model. Its data is synthetic. In Training Studio, supply your own splits and model chat template, then start with a short check run.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Run the experiment

Run cells in order. Edit the code; Shift + Enter runs the current cell.

In [ ]:
class TinyLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(16, 24)
        self.position = nn.Embedding(8, 24)
        self.block = nn.TransformerEncoderLayer(24, 4, 48, dropout=0, batch_first=True)
        self.head = nn.Linear(24, 16)
    def forward(self, ids):
        length = ids.shape[1]
        h = self.embedding(ids) + self.position(torch.arange(length))
        causal = torch.triu(torch.ones(length, length, dtype=torch.bool), diagonal=1)
        return self.head(self.block(h, src_mask=causal))
model = TinyLM(); optimizer = torch.optim.AdamW(model.parameters(), lr=0.02)
batch = torch.tensor([[1, 2, 3, 4, 5, 6, 7, 8]]).repeat(8, 1)
labels = batch[:, 1:].clone(); labels[:, :3] = -100
losses = []
for step in range(35):
    optimizer.zero_grad(set_to_none=True)
    loss = F.cross_entropy(model(batch[:, :-1]).flatten(0, 1), labels.flatten())
    loss.backward(); optimizer.step(); losses.append(loss.item())
print('supervised tokens per sample:', (labels[0] != -100).sum().item())
plt.plot(losses, color='#d73535'); plt.ylabel('response-only CE'); plt.show()

## Try it yourself

Remove the prompt mask and compare loss. Change prompt length while preserving shifted labels. Why does memorizing one sequence not establish useful SFT?

Save a notebook copy and record what you changed, expected, and measured.

## Continue in Studio

The workspace link above the lesson opens related tools. This notebook does not launch large training jobs. Before real training, check data, resources, and method support for the chosen model.

## Sources

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)